# Declarative attention below the scale where declaring works

Declarative Attention (arXiv:2609.02737) has a model state which region of context it must
read, and an inference engine skip the rest of the KV cache. Reported on 27-31B models,
zero-shot, 52.0% / 31.1% fewer attended tokens.

This asks whether the declaration is reliable at 0.5-1.5B. Two elicitations: **generate**
(`FOCUS: <k>`, as published) and **read** (per-region keep score off the logits).

Gold regions are permuted per probe, so the best constant policy sits at 1/K. On CPU at
360M both modes land at or below chance; the discriminating measurement needs 1.5B and
n in the hundreds.

In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'   # cleared only if you prefer a Kaggle Dataset
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

if not looks_like_source(REPO):
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)
    if not looks_like_source(REPO) and GIT_URL:
        r = subprocess.run(['git', 'clone', GIT_URL, REPO], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert looks_like_source(REPO), (
        'No source found. Either (a) run scripts/package_source.py locally, upload the zip '
        'as a Kaggle Dataset, and attach it via Add Input, or (b) set GIT_URL above. '
        f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])

# peft raises rather than degrading when it finds an old torchao, and Kaggle ships
# 0.10.0 against a >0.16 requirement. Nothing here uses torchao, so remove it.
try:
    import importlib.metadata as _md
    _v = _md.version('torchao')
    if tuple(int(x) for x in _v.split('.')[:2]) < (0, 16):
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchao'])
        print(f'removed incompatible torchao {_v}')
except Exception:
    pass
exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

QUICK = False
CFG = 'configs/kaggle_declare.yaml'
RUNS = '/kaggle/working/artifacts/runs_declare'
N_EVAL = 8 if QUICK else 64
N_PROBES = 24 if QUICK else 0        # 0 = every probe
MODELS = ['Qwen/Qwen2.5-0.5B-Instruct'] if QUICK else [
    'Qwen/Qwen2.5-0.5B-Instruct',
    'Qwen/Qwen2.5-1.5B-Instruct',
]
print(f'QUICK={QUICK}  models={len(MODELS)}  eval trajectories={N_EVAL}')
print('read mode costs K=8 forward passes per probe; generate costs 1 generation.')
print('Both run twice per probe, once shuffled, for the slot-stability test.')


## Data

Supporting paragraphs spread across the whole trajectory, so position carries no prior.

In [ ]:
run(f"python data/load_hotpotqa.py --n-train 8 --n-eval {N_EVAL} --per-trajectory 4 --early-frac 1.0 --out artifacts/data/hotpotqa_spread")


## Sweep

Every model sees identical layouts, so `generate` and `read` are directly comparable.

In [ ]:
lim = f'--limit {N_PROBES}' if N_PROBES else ''
for i, m in enumerate(MODELS, 1):
    slug = m.split('/')[-1]
    print('=' * 70)
    print(f'[{i}/{len(MODELS)}] {slug}', flush=True)
    run(f"python declare/run.py --config {CFG} --modes generate read {lim} --out {RUNS}/{slug} --set model.base={m}")


In [ ]:
import json, glob
paths = sorted(glob.glob(f'{RUNS}/*/summary_all.json'))
print(f'found {len(paths)} runs')
hdr = f"{'model':<28} {'mode':<9} {'hit':>6} {'rand':>6} {'const':>6} {'slot':>6} {'modal':>6} {'unpar':>6}"
print(hdr)
for p in paths:
    for s in json.load(open(p)):
        name = s['model'].split('/')[-1]
        print(f"{name:<28} {s['label']:<9} {s['hit_rate']:6.3f} {s['random_control']:6.3f} "
              f"{s['best_constant_control']:6.3f} {s.get('slot_stable_rate', 0):6.3f} "
              f"{s['modal_share']:6.3f} {s['unparsed_rate']:6.3f}")
print()
print('hit must beat both rand and const. slot near 1.0 means the model names a slot,')
print('not content - the failure this arm exists to measure.')


In [ ]:
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs_declare.zip")
